In [3]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset, Audio
import evaluate
import torch

/info/etu/m1/s2506992/miniconda3/envs/asr2526_new/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mls = load_dataset("parler-tts/mls_eng", split="dev",streaming="true")

In [3]:
mls = mls.cast_column("audio", Audio(sampling_rate=16_000))

In [4]:
print(type(mls))

<class 'datasets.iterable_dataset.IterableDataset'>


In [8]:
print(mls.features)

{'audio': Audio(sampling_rate=16000, decode=True, stream_index=None), 'original_path': Value('string'), 'begin_time': Value('float64'), 'end_time': Value('float64'), 'transcript': Value('string'), 'audio_duration': Value('float64'), 'speaker_id': Value('string'), 'book_id': Value('string')}


In [9]:
print(mls['transcript'])

In [10]:
print(mls['audio'])

In [11]:
input_speech = next(iter(mls))

In [29]:
print(type(input_speech))

<class 'dict'>


In [12]:
print(input_speech)

{'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7f89af0f9ff0>, 'original_path': 'http://www.archive.org/download/birdsvol5feb1899_1510_librivox/birdsandallnaturefeb1899_14_various_64kb.mp3', 'begin_time': 32.46, 'end_time': 44.86, 'transcript': 'but the owl is not a burglar he is the friend of man there is no other bird that does the farmer so much good as the owl the owl comes out in the dark to get the small animals that are out at that time stealing things from the farmer', 'audio_duration': 12.4, 'speaker_id': '10214', 'book_id': '10108'}


In [14]:
print(input_speech.keys())

dict_keys(['audio', 'original_path', 'begin_time', 'end_time', 'transcript', 'audio_duration', 'speaker_id', 'book_id'])


In [16]:
signal_audio = input_speech['audio']['array']

In [17]:
print(type(signal_audio))

<class 'numpy.ndarray'>


In [18]:
processor = WhisperProcessor.from_pretrained("openai/whisper-small")

In [19]:
input_features = processor( signal_audio, sampling_rate=input_speech['audio']["sampling_rate"], return_tensors="pt").input_features

In [20]:
print(type(input_features))        # <class 'list'>
print(len(input_features))         # 1 (si un seul audio)
print(type(input_features[0]))

<class 'torch.Tensor'>
1
<class 'torch.Tensor'>


In [21]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
forced_decoder_ids = processor.get_decoder_prompt_ids(language="en", task="transcribe")

In [22]:
predicted_ids = model.generate(input_features, forced_decoder_ids=forced_decoder_ids)

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [23]:
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
label = input_speech['transcript']

In [26]:
print("whisper: ",transcription)
print("la verite: ",label)

whisper:   But the owl is not a burglar. He is the friend of man. There is no other bird that does the farmer so much good as the owl. The owl comes out in the dark to get the small animals that are out at that time stealing things from the farmer.
la verite:  but the owl is not a burglar he is the friend of man there is no other bird that does the farmer so much good as the owl the owl comes out in the dark to get the small animals that are out at that time stealing things from the farmer


In [28]:
for example in mls:
    print(type(example))

<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'dict'>
<class 'di

In [12]:
def whisper_inference():
    # importer les metriques
    wer_metric = evaluate.load("wer")
    cer_metric = evaluate.load("cer")

    # pour execution sur gpu si dispo
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)

    #le necessaire pour inferer, preparer le processeur (extracteur des caracteristiques acoustiques) et le decodeur ext ..
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
    model = model.to(device)
    forced_decoder_ids = processor.get_decoder_prompt_ids(language="english", task='transcribe')

    # Loader le dataset
    mls = load_dataset("parler-tts/mls_eng", split="dev",streaming=True)
    # Modifier la frequence des audios en 16000khz
    mls = mls.cast_column("audio", Audio(sampling_rate=16_000))

    results = []

    compteur = 0
    # Iterer sur le dataset pour inferer
    for signal in mls:

        input_features = processor( signal['audio']['array'], sampling_rate=signal['audio']['sampling_rate'], return_tensors="pt").input_features
        predicted_ids = model.generate(input_features = input_features.to(model.device), forced_decoder_ids=forced_decoder_ids)
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        # Stocker les resultats
        results.append({
            "sentence": signal['transcript'],  
            "predicted": transcription 
        })
        compteur = compteur + 1
        if compteur == 700: 
            break 
        elif compteur % 50 == 0: 
            print(compteur)
        
    references = [r["sentence"] for r in results]
    predictions = [r["predicted"] for r in results]

    # Calculer les metriques
    wer = 100 * wer_metric.compute(references=references, predictions=predictions)
    cer = 100 * cer_metric.compute(references=references, predictions=predictions)   

    return wer, cer

In [ ]:
wer, cer= whisper_inference()

cuda


In [ ]:
print("wer",wer)
print("cer",cer)

In [3]:
mls_test = load_dataset("parler-tts/mls_eng", split="test",streaming="true")

In [4]:
mls_french_test = load_dataset("facebook/multilingual_librispeech", "french", split="test", streaming="true")

In [5]:
dataset = load_dataset("issai/Turkish_Speech_Corpus", split="test[:500]", streaming="true")

ValueError: The TAR archives of the dataset should be in WebDataset format, but the files in the archive don't share the same prefix or the same types.

In [8]:
# Essayer avec turkishneuralvoice
dataset = load_dataset("erenfazlioglu/turkishneuralvoice", split="train", streaming="true")

In [9]:
dataset.features

{'audio': Audio(sampling_rate=None, decode=True, stream_index=None),
 'transcription': Value('string')}

In [4]:
def whisper_inference(corpus, langue, dataset):
    # importer les metriques
    wer_metric = evaluate.load("wer")
    cer_metric = evaluate.load("cer")
    # pour execution sur gpu si dispo
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    #le necessaire pour inferer, preparer le processeur (extracteur des caracteristiques acoustiques) et le decodeur etc..
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
    model = model.to(device)
    forced_decoder_ids = processor.get_decoder_prompt_ids(language=langue, task='transcribe')
    
    # Loader le dataset
    mls = load_dataset(dataset, split=corpus, streaming=True)
    # Modifier la frequence des audios en 16000khz
    mls = mls.cast_column("audio", Audio(sampling_rate=16_000))
    
    results = []
    compteur = 0
    
    # Iterer sur le dataset pour inferer
    for signal in mls:
        input_features = processor(
            signal['audio']['array'], 
            sampling_rate=signal['audio']['sampling_rate'], 
            return_tensors="pt"
        ).input_features
        
        predicted_ids = model.generate(
            input_features=input_features.to(device), 
            forced_decoder_ids=forced_decoder_ids
        )
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        
        # Stocker les resultats
        results.append({
            "sentence": signal['transcript'],  
            "predicted": transcription 
        })
        
        compteur += 1
        
        # Afficher la progression
        if compteur % 50 == 0:
            print(f"Processing: {compteur}/700 samples... pour la langue: {langue} ... sur le corpus: {corpus}")
        
        # Arreter a 700
        if compteur == 700: 
            break
    
    references = [r["sentence"] for r in results]
    predictions = [r["predicted"] for r in results]
    
    # Calculer les metriques
    wer = 100 * wer_metric.compute(references=references, predictions=predictions)
    cer = 100 * cer_metric.compute(references=references, predictions=predictions)
    
    print(f"\n=== Results for {langue} ===")
    print(f"Dataset: {dataset}")
    print(f"Split: {corpus}")
    print(f"Samples evaluated: {len(results)}")
    print(f"WER: {wer:.2f}%")
    print(f"CER: {cer:.2f}%")
    
    return wer, cer

In [5]:
corpus = "dev"
langue = "english"
dataset = "parler-tts/mls_eng"

In [6]:
wer, cer = whisper_inference(corpus, langue, dataset)

Using device: cuda


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Processing: 50/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 100/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 150/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 200/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 250/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 300/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 350/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 400/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 450/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 500/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 550/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 600/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 650/700 samples... pour la langue: english ... sur le

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
=== Results for english ===
Dataset: parler-tts/mls_eng
Split: dev
Samples evaluated: 700
WER: 27.90%
CER: 8.55%

SyntaxError: invalid syntax (3678026514.py, line 1)

In [8]:
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

In [9]:
def whisper_inference(corpus, langue, dataset):
    
    # importer les metriques
    wer_metric = evaluate.load("wer")
    cer_metric = evaluate.load("cer")
    normalizer = BasicTextNormalizer()
    
    # pour execution sur gpu si dispo
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    #le necessaire pour inferer, preparer le processeur (extracteur des caracteristiques acoustiques) et le decodeur etc..
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
    model = model.to(device)
    forced_decoder_ids = processor.get_decoder_prompt_ids(language=langue, task='transcribe')
    
    # Loader le dataset
    mls = load_dataset(dataset, split=corpus, streaming=True)
    # Modifier la frequence des audios en 16000khz
    mls = mls.cast_column("audio", Audio(sampling_rate=16_000))
    
    results = []
    compteur = 0
    
    # Iterer sur le dataset pour inferer
    for signal in mls:
        input_features = processor(
            signal['audio']['array'], 
            sampling_rate=signal['audio']['sampling_rate'], 
            return_tensors="pt"
        ).input_features
        
        predicted_ids = model.generate(
            input_features=input_features.to(device), 
            forced_decoder_ids=forced_decoder_ids
        )
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        
        # Stocker les resultats
        results.append({
            "sentence": normalizer(signal['transcript']),  
            "predicted": normalizer(transcription)
        })
        
        compteur += 1
        
        # Afficher la progression
        if compteur % 50 == 0:
            print(f"Processing: {compteur}/700 samples... pour la langue: {langue} ... sur le corpus: {corpus}")
        
        # Arreter a 700
        if compteur == 700: 
            break
    
    references = [r["sentence"] for r in results]
    predictions = [r["predicted"] for r in results]
    
    # Calculer les metriques
    wer = 100 * wer_metric.compute(references=references, predictions=predictions)
    cer = 100 * cer_metric.compute(references=references, predictions=predictions)
    
    print(f"\n=== Results for {langue} ===")
    print(f"Dataset: {dataset}")
    print(f"Split: {corpus}")
    print(f"Samples evaluated: {len(results)}")
    print(f"WER: {wer:.2f}%")
    print(f"CER: {cer:.2f}%")
    
    return wer, cer

In [10]:
corpus = "dev"
langue = "english"
dataset = "parler-tts/mls_eng"

In [11]:
wer, cer = whisper_inference(corpus, langue, dataset)

Using device: cuda
Processing: 50/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 100/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 150/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 200/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 250/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 300/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 350/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 400/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 450/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 500/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 550/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 600/700 samples... pour la langue: english ... sur le corpus: dev
Processing: 650/700 samples... pour la langue:

In [12]:
fleur =load_dataset("google/fleurs", "fr_fr", split="test", streaming=True)

RuntimeError: Dataset scripts are no longer supported, but found fleurs.py

In [4]:
dataset = load_dataset("parler-tts/mls_eng", split="dev",streaming="true")

In [5]:
print(f"Version: {dataset.version if hasattr(dataset, 'version') else 'N/A'}")

Version: 0.0.0
